# 01 — Corpus Collection

Everything the pipeline learns comes from the ARO source tree. This notebook
walks the repository and assembles a single manifest that captures:

- **`.aro` files** from `Examples/` and `ARO-Application/` — real, working programs
- **Proposals** — the authoritative language specification (59 documents)
- **Book chapters** — the full developer guide across six books
- **Swift action files** — source of truth for every action's verbs and prepositions
- **Syntax rules** — extracted from `CLAUDE.md`, the authoritative maintained source
- **Live CLI output** — `aro actions`, `aro examples` from the actual binary

**Output:** `../data/01_corpus/manifest.json`

In [10]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import os
import json
import subprocess
from pathlib import Path

ARO_APP_DIR = (SCRIPT_DIR / '../../../ARO-Application/').resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'ARO root:        {ARO_ROOT}')
print(f'ARO-Application: {ARO_APP_DIR}  (exists: {ARO_APP_DIR.exists()})')
print(f'Output:          {DATA_DIR}')

TRAIN_ON_BASE=True → using base model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Config loaded | model=mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
ARO root:  /Users/kris/Projects/ARO/ARO-Train
Pairs:     /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl
Adapters:  /Users/kris/Projects/ARO/ARO-Train/Train/data/adapters
ARO root:        /Users/kris/Projects/ARO/ARO-Train
ARO-Application: /Users/kris/Projects/ARO/ARO-Application  (exists: True)
Output:          /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge


## Check aro CLI

In [11]:
def run_aro(args, timeout=30):
    """Run an aro CLI command and return (stdout, stderr, returncode).
    
    Returns empty string for stdout when the command fails, and prints
    a warning so failures are visible instead of silently propagated.
    """
    try:
        r = subprocess.run(['aro'] + args, capture_output=True, text=True, timeout=timeout)
        if r.returncode != 0:
            print(f'  ⚠  aro {" ".join(args)} failed (exit {r.returncode}): {r.stderr.strip()[:120]}')
            return '', r.stderr.strip(), r.returncode
        return r.stdout.strip(), r.stderr.strip(), r.returncode
    except FileNotFoundError:
        print('  ⚠  aro binary not found in PATH')
        return '', 'aro binary not found in PATH', -1
    except subprocess.TimeoutExpired:
        print(f'  ⚠  aro {" ".join(args)} timed out after {timeout}s')
        return '', 'timeout', -1

out, err, code = run_aro(['--version'])
print(f'aro: {out or err}')

aro: 0.9.2


## Collect .aro files

In [12]:
aro_files = []
search_roots = [ARO_ROOT / 'Examples']
if ARO_APP_DIR.exists():
    search_roots.append(ARO_APP_DIR)

for root in search_roots:
    found = list(root.rglob('*.aro'))
    print(f'  {root.name}: {len(found)} .aro files')
    aro_files.extend(found)

print(f'Total: {len(aro_files)} .aro files')

  Examples: 127 .aro files
  ARO-Application: 34 .aro files
Total: 161 .aro files


## Collect Proposals, Book, Swift action files

In [13]:
proposals     = sorted((ARO_ROOT / 'Proposals').glob('*.md'))
book_chapters = sorted((ARO_ROOT / 'Book').rglob('*.md')) if (ARO_ROOT / 'Book').exists() else []
swift_actions = sorted((ARO_ROOT / 'Sources' / 'ARORuntime' / 'Actions').rglob('*.swift'))

print(f'Proposals:     {len(proposals)}')
print(f'Book chapters: {len(book_chapters)}')
print(f'Swift actions: {len(swift_actions)}')

Proposals:     61
Book chapters: 133
Swift actions: 33


## Extract syntax from CLAUDE.md + fetch CLI data

The `aro` CLI has no `syntax` subcommand — the previous code silently stored an
error string.  Instead, we extract syntax rules directly from `CLAUDE.md`, which
is the authoritative, maintained source.  Actions and examples are still fetched
from the live CLI (`aro actions`, `aro examples`).

In [14]:
import re as _re

# ── Generate aro_syntax from CLAUDE.md ────────────────────────────────────────
# The `aro syntax` CLI subcommand does not exist; the old code silently stored
# the resulting error string as the syntax summary, poisoning every downstream
# system prompt.  Instead, extract syntax rules directly from CLAUDE.md which
# is the authoritative source maintained alongside the code.

_claude_md_path = ARO_ROOT / 'CLAUDE.md'
_claude_md = _claude_md_path.read_text() if _claude_md_path.exists() else ''

def _extract_syntax_summary(claude_md: str) -> str:
    """Build a concise ARO syntax summary from CLAUDE.md content."""
    sections = []

    # 1. Core syntax block (```aro ... ```)
    aro_blocks = _re.findall(r'```aro\n(.*?)```', claude_md, _re.DOTALL)
    if aro_blocks:
        sections.append('## Core Syntax Examples\n' + '\n---\n'.join(b.strip() for b in aro_blocks[:6]))

    # 2. Extract the "ARO Syntax" section if present
    m = _re.search(r'## ARO Syntax\s*\n(.*?)(?=\n## |\Z)', claude_md, _re.DOTALL)
    if m:
        sections.append(m.group(1).strip())

    # 3. Extract the Computations table
    m = _re.search(r'### Computations\s*\n(.*?)(?=\n### |\n## |\Z)', claude_md, _re.DOTALL)
    if m:
        sections.append('## Computations\n' + m.group(1).strip())

    # 4. Extract the event-driven feature sets table
    m = _re.search(r'### Event-Driven Feature Sets\s*\n(.*?)(?=\n### |\n## |\Z)', claude_md, _re.DOTALL)
    if m:
        sections.append('## Event-Driven Feature Sets\n' + m.group(1).strip())

    # 5. Extract key rules from various sections
    rules = []
    for pattern in [
        r'An ARO application is a \*\*directory\*\*.*?(?=\n## |\n### )',
        r'\*\*Key Rules:\*\*\s*\n(.*?)(?=\n## |\n### )',
        r'### Happy Case\s*\n(.*?)(?=\n### |\n## |\Z)',
        r'### Long-Running Applications\s*\n(.*?)(?=\n### |\n## |\Z)',
    ]:
        m = _re.search(pattern, claude_md, _re.DOTALL)
        if m:
            rules.append(m.group(0).strip() if m.lastindex is None else m.group(1).strip())
    if rules:
        sections.append('## Key Rules\n' + '\n\n'.join(rules))

    # 6. Action semantic roles
    m = _re.search(r'### Action Semantic Roles\s*\n(.*?)(?=\n### |\n## |\Z)', claude_md, _re.DOTALL)
    if m:
        sections.append('## Action Semantic Roles\n' + m.group(1).strip())

    # 7. Contract-first HTTP
    m = _re.search(r'### Contract-First HTTP.*?\n(.*?)(?=\n### |\n## |\Z)', claude_md, _re.DOTALL)
    if m:
        sections.append('## Contract-First HTTP APIs\n' + m.group(1).strip()[:1500])

    result = '\n\n'.join(sections)
    return result[:5000]  # cap at 5000 chars for system prompt use


syntax_out = _extract_syntax_summary(_claude_md)
if len(syntax_out) < 200:
    print(f'⚠  WARNING: aro_syntax extraction from CLAUDE.md produced only {len(syntax_out)} chars.')
    print('   Check that CLAUDE.md contains the expected ARO syntax documentation.')
else:
    print(f'✓  aro_syntax extracted from CLAUDE.md: {len(syntax_out):,} chars')

# ── Fetch actions and examples from CLI (these commands DO exist) ─────────────
print('Calling aro CLI for actions and examples...')
actions_out,  _, _ = run_aro(['actions'])
examples_out, _, _ = run_aro(['examples'])

print(f'aro actions:  {len(actions_out):,} chars')
print(f'aro examples: {len(examples_out):,} chars')

✓  aro_syntax extracted from CLAUDE.md: 5,000 chars
Calling aro CLI for actions and examples...
  ⚠  aro examples failed (exit 1): 
aro actions:  2,768 chars
aro examples: 0 chars


## Read static reference docs

In [15]:
extra_docs = {}
for doc_path in [ARO_ROOT / 'README.md', ARO_ROOT / 'CLAUDE.md', ARO_ROOT / 'OVERVIEW.md']:
    if doc_path.exists():
        extra_docs[doc_path.name] = doc_path.read_text()
        print(f'  {doc_path.name}: {len(extra_docs[doc_path.name]):,} chars')

  README.md: 12,685 chars
  CLAUDE.md: 24,246 chars
  OVERVIEW.md: 13,053 chars


## Save manifest

In [16]:
manifest = {
    'aro_files':     [str(f) for f in aro_files],
    'proposals':     [str(f) for f in proposals],
    'book_chapters': [str(f) for f in book_chapters],
    'swift_actions': [str(f) for f in swift_actions],
    'extra_docs':    {k: v[:60_000] for k, v in extra_docs.items()},
    'aro_syntax':    syntax_out,
    'aro_actions':   actions_out,
    'aro_examples':  examples_out,
}

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

total = len(aro_files) + len(proposals) + len(book_chapters) + len(swift_actions)
print(f'Manifest saved: {manifest_path}')
print(f'Total files indexed: {total}')

Manifest saved: /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/manifest.json
Total files indexed: 388


In [17]:
import csv
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_csv_path = _run_dir / '01_corpus_collection.csv'

with open(_csv_path, 'w', newline='') as _f:
    _w = csv.writer(_f)
    _w.writerow(['source_type', 'file_path', 'char_count'])
    for _fp in aro_files:
        _w.writerow(['aro_file', str(_fp), len(Path(_fp).read_text())])
    for _fp in proposals:
        _w.writerow(['proposal', str(_fp), len(Path(_fp).read_text())])
    for _fp in book_chapters:
        _w.writerow(['book_chapter', str(_fp), len(Path(_fp).read_text())])
    for _fp in swift_actions:
        _w.writerow(['swift_action', str(_fp), len(Path(_fp).read_text())])
    for _name, _content in extra_docs.items():
        _w.writerow(['reference_doc', _name, len(_content)])

print(f'CSV saved: {_csv_path}  ({total + len(extra_docs)} rows)')

CSV saved: run/2026-04-25/01_corpus_collection.csv  (391 rows)


In [18]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '01_corpus_collection.png'

_cats   = ['ARO files', 'Proposals', 'Book chapters', 'Swift actions', 'Reference docs']
_vals   = [len(aro_files), len(proposals), len(book_chapters), len(swift_actions), len(extra_docs)]
_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(_cats, _vals, color=_colors, edgecolor='white', width=0.6)
for bar, v in zip(bars, _vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(v), ha='center', va='bottom', fontsize=12, fontweight='bold', color='#2c3e50')
ax.set_ylabel('Items collected')
ax.set_title(f'Corpus Collection — {sum(_vals):,} items across {len(_cats)} categories',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(_vals) * 1.18)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

Saved: run/2026-04-25/01_corpus_collection.png
